In [11]:
import json
from fastcoref import FCoref
from collections import defaultdict

In [12]:
model = FCoref(device='cuda:0')

04/13/2026 17:47:38 - INFO - 	 missing_keys: []
04/13/2026 17:47:38 - INFO - 	 unexpected_keys: []
04/13/2026 17:47:38 - INFO - 	 mismatched_keys: []
04/13/2026 17:47:38 - INFO - 	 error_msgs: []
04/13/2026 17:47:38 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [41]:
# file = "DATA/BASIL-main/articles/2019/0d10341a-9dba-4374-a524-814c300d1611_1.json"
file = "DATA/BASIL-main/articles/2016/7a97de89-1433-46f7-bcc2-f80b041cb9a0_2.json"
# file = "DATA/BASIL-main/articles/2016/d4a4bc34-e48b-4485-86b2-f1daf4d464dd_1.json"
# file = "DATA/BASIL-main/articles/2015/2b9468af-4ab0-4f25-85f4-c3ef93d72006_1.json"
def to_article_BASIL(filename: str) -> list[str]:
    ret: list[str] = []
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    #Export from nested list to a single list of sentences
    for paragraph in data["body-paragraphs"]:
        if len(paragraph) == 1:
            ret.append(paragraph[0] + " ")
        else:
            for sentence in paragraph:
                ret.append(sentence + " ") 

    return ret
Sentence_list = to_article_BASIL(file)
#get only the first ten sentences
text = "".join(Sentence_list)
text

'AUGUSTA, Me. — Paul LePage, the embattled Republican governor of Maine, declared on Wednesday that he would not step down despite widespread criticism over a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term. “I will not resign,” said Mr. LePage, who tried to put to rest swirling questions about his state of mind. “I’m not an alcoholic, and I’m not a drug addict, and I don’t have mental issues,” he said. “What I have is a backbone.” Mr. LePage nevertheless said he was seeking “spiritual guidance,” had apologized for his threat, and vowed to make one change to his behavior: “I will no longer speak to the press ever again after today,” he said. Behind the scenes, state lawmakers — Democrats and Senate Republicans — exasperated after six years of political controversies and what they saw as erratic behavior, were scrambling to figure out what, if anything, they could do to formally address this latest cris

In [65]:
preds = model.predict(texts=text)

04/13/2026 21:57:25 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 29.72 examples/s]
04/13/2026 21:57:25 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00, 53.55it/s]


In [29]:
EL_dict = {181962: {'name': 'Augusta',
  'text': {'AUGUSTA'},
  'description': 'consolidated city-county in Richmond County, Georgia, United States'},
 619989: {'name': 'The Menagerie',
  'text': {'Me'},
  'description': 'two-part episode of Star Trek: The Original Series'},
 172008: {'name': 'full stop',
  'text': {'.'},
  'description': 'punctuation to signal the end of a sentence'},
 881196: {'name': 'Paul LePage',
  'text': {'Paul LePage'},
  'description': 'American businessman, Republican Party politician, and the 74th Governor of Maine'},
 724: {'name': 'Maine',
  'text': {'Maine'},
  'description': 'state of the United States of America'},
 29552: {'name': 'Democratic Party',
  'text': {'Democrats'},
  'description': 'political party in the United States'},
 29468: {'name': 'Republican Party',
  'text': {'Republicans'},
  'description': 'political party in the United States'},
 14948800: {'name': 'Kenneth Fredette',
  'text': {'Kenneth Fredette'},
  'description': 'American politician'},
 6834848: {'name': 'Michael Thibodeau',
  'text': {'Michael Thibodeau'},
  'description': 'Member of Maine State Senate'},
 7358441: {'name': 'Roger Katz',
  'text': {'Roger Katz'},
  'description': 'Maine State Senator'},
 55668409: {'name': 'Westfield Republican',
  'text': {'Republican'},
  'description': 'newspaper published in Westfield, New York'},
 16147040: {'name': 'Justin Alfond',
  'text': {'Justin Alfond'},
  'description': 'American politician'},
 26721176: {'name': 'Drew Gattine',
  'text': {'Drew Gattine'},
  'description': 'American politician'}}

In [39]:
import re
def regex_match_score(pattern: str, text: str):
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        return 0.0
    
    matched_len = len(match.group())
    pattern_len = len(pattern)
    
    return matched_len / pattern_len

In [64]:
# master_zip = zip([[item for item in sub if item.lower() not in pronouns] for sub in cluster], spans)
cluster_text = [[item for item in sub if item.lower() not in pronouns] for sub in cluster]
name_list = []
# for entity in EL_dict.values():
#     name_list.append(entity['name'])



for coRef in cluster_text:
    match = re.search( "Maine", "".join(coRef), re.IGNORECASE)
    
    if not match:
        print("CoRef: ", "".join(coRef), "Score :" , 0.0 , "\n")
        continue
        
    match_len = len(match.group())
    pattern_len = len("Maine")
        
    print("CoRef: ", "".join(coRef), "Score", match_len/pattern_len, "\n" )
    
cluster_text


CoRef:  Me.Mainethe statethe state’sMaineThe state’sMaine’s Score 1.0 

CoRef:  Paul LePage, the embattled Republican governor of Maine,Mr. LePage, who tried to put to rest swirling questions about his state of mindMr. LePageMr. LePageMr. LePageThe governorMr. LePagethe governorMr. LePage’sMr. LePage’sthe governorMr. LePageMr. LePageMr. LePage’sthe governor’sMr. LePage’sMr. LePageMr. LePageMr. LePageMr. LePageMr. LePage Score 1.0 

CoRef:  a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second termhis threathis latest tirade Score : 0.0 

CoRef:   Score : 0.0 

CoRef:  a Democratic lawmakerthe lawmakerRepresentative Fredette’s Score : 0.0 

CoRef:  sweeping statements linking minorities to the state’s drug crisisthe remarks Score : 0.0 

CoRef:  The state’s four legislative caucusesall four Score : 0.0 

CoRef:  TuesdayTuesday Score : 0.0 

CoRef:  Mr. LePage’s Republican allies in the House, including the minor

[['Me.',
  'Maine',
  'the state',
  'the state’s',
  'Maine',
  'The state’s',
  'Maine’s'],
 ['Paul LePage, the embattled Republican governor of Maine,',
  'Mr. LePage, who tried to put to rest swirling questions about his state of mind',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage',
  'The governor',
  'Mr. LePage',
  'the governor',
  'Mr. LePage’s',
  'Mr. LePage’s',
  'the governor',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage’s',
  'the governor’s',
  'Mr. LePage’s',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage'],
 ['a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term',
  'his threat',
  'his latest tirade'],
 [],
 ['a Democratic lawmaker', 'the lawmaker', 'Representative Fredette’s'],
 ['sweeping statements linking minorities to the state’s drug crisis',
  'the remarks'],
 ['The state’s four legislative caucuses', 'all four'],
 ['Tuesday', 'Tuesday'],
 ['Mr. L

In [ ]:
cluster



[['Me.',
  'Maine',
  'the state',
  'the state’s',
  'Maine',
  'The state’s',
  'Maine’s'],
 ['Paul LePage, the embattled Republican governor of Maine,',
  'he',
  'him',
  'he',
  'his',
  'I',
  'Mr. LePage, who tried to put to rest swirling questions about his state of mind',
  'his',
  'I',
  'I',
  'I',
  'he',
  'I',
  'Mr. LePage',
  'he',
  'his',
  'his',
  'I',
  'he',
  'Mr. LePage',
  'him',
  'Mr. LePage',
  'him',
  'The governor',
  'he',
  'Mr. LePage',
  'his',
  'the governor',
  'his',
  'Mr. LePage’s',
  'him',
  'Mr. LePage’s',
  'the governor',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage’s',
  'the governor’s',
  'Mr. LePage’s',
  'he',
  'his',
  'his',
  'Mr. LePage',
  'I',
  'Mr. LePage',
  'his',
  'I',
  'Mr. LePage',
  'he',
  'Mr. LePage',
  'he',
  'he',
  'I',
  'Mr. LePage',
  'he'],
 ['a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term',
  'his threat',
  'his lates

In [9]:
spans

[[(15, 74),
  (102, 104),
  (235, 238),
  (263, 265),
  (278, 281),
  (296, 297),
  (321, 400),
  (383, 386),
  (403, 404),
  (429, 430),
  (456, 457),
  (485, 487),
  (500, 501),
  (523, 533),
  (552, 554),
  (608, 611),
  (652, 655),
  (667, 668),
  (728, 730),
  (1083, 1093),
  (1187, 1190),
  (1200, 1210),
  (1265, 1268),
  (1279, 1291),
  (1302, 1305),
  (1350, 1352),
  (1441, 1451),
  (1468, 1471),
  (1706, 1718),
  (1724, 1727),
  (1893, 1905),
  (2058, 2061),
  (2779, 2791),
  (2874, 2886),
  (3267, 3277),
  (3755, 3765),
  (3884, 3896),
  (4090, 4104),
  (4198, 4210),
  (4330, 4332),
  (4407, 4410),
  (4451, 4454),
  (4528, 4538),
  (4597, 4598),
  (4628, 4638),
  (4698, 4701),
  (4740, 4741),
  (4775, 4785),
  (4806, 4808),
  (4982, 4992),
  (5044, 5046),
  (5084, 5086),
  (5164, 5165),
  (5205, 5215),
  (5234, 5236)],
 [(159, 293), (608, 618), (1724, 1741)],
 [(756, 806), (873, 877), (952, 956)],
 [(68, 73),
  (1044, 1053),
  (1141, 1152),
  (1572, 1577),
  (1743, 1754),
  (

In [66]:
cluster = preds.get_clusters()
spans: list[str] = preds.get_clusters(as_strings=False)

pronouns = {
    "i", "me", "my", "mine", "myself", "we", "us", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself",
    "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", 
    "their", "theirs", "themselves", "who", "whom", "whose", "which", "what",
    "this", "that", "these", "those", "anyone", "someone", "no one", "everyone"
}


def create_dictionary( cluster, spans):
    my_dict = defaultdict(list)
    for nouns, indexs in zip(cluster, spans):
        for noun in nouns:
            if noun not in pronouns:
                key = noun
                break
        my_dict[key].extend(indexs)
    
    return my_dict




my_dict = create_dictionary(cluster, spans)


for key in my_dict:
    print(key)




Me.
Paul LePage, the embattled Republican governor of Maine,
a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term
a Democratic lawmaker
sweeping statements linking minorities to the state’s drug crisis
The state’s four legislative caucuses
Tuesday
Mr. LePage’s Republican allies in the House, including the minority leader, Kenneth Fredette,
a special session
announced
Wednesday
some other state Republicans
the president of the Senate, whose members govern larger and more competitive districts,
the House
The Republican senate caucus
the Republican leader, Michael Thibodeau,
Senate
many Senate Republicans
The controversy, which has galvanized the state,
be
lawmakers like Senator Roger Katz, a Republican who has been critical of Mr. LePage,
Senator Roger Katz, a Republican who has been critical of Mr. LePage,
convene
Republicans in the House
Democrats
the political storm
Wednesday morning


In [76]:
def coRef_to_raw_text( cluster):
    temp = []
    for list in cluster:
        temp.extend(list)
    return " ".join(temp)
coref_text = coRef_to_raw_text(cluster)
coref_text

'Me. Maine the state the state’s Maine The state’s Maine’s Paul LePage, the embattled Republican governor of Maine, he him he his I Mr. LePage, who tried to put to rest swirling questions about his state of mind his I I I he I Mr. LePage he his his I he Mr. LePage him Mr. LePage him The governor he Mr. LePage his the governor his Mr. LePage’s him Mr. LePage’s the governor Mr. LePage Mr. LePage Mr. LePage’s the governor’s Mr. LePage’s he his his Mr. LePage I Mr. LePage his I Mr. LePage he Mr. LePage he he I Mr. LePage he a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term his threat his latest tirade they they a Democratic lawmaker him the lawmaker Representative Fredette’s sweeping statements linking minorities to the state’s drug crisis the remarks The state’s four legislative caucuses they all four Tuesday Tuesday Mr. LePage’s Republican allies in the House, including the minority leader, Kenneth Frede

In [71]:
cluster

[['Me.',
  'Maine',
  'the state',
  'the state’s',
  'Maine',
  'The state’s',
  'Maine’s'],
 ['Paul LePage, the embattled Republican governor of Maine,',
  'he',
  'him',
  'he',
  'his',
  'I',
  'Mr. LePage, who tried to put to rest swirling questions about his state of mind',
  'his',
  'I',
  'I',
  'I',
  'he',
  'I',
  'Mr. LePage',
  'he',
  'his',
  'his',
  'I',
  'he',
  'Mr. LePage',
  'him',
  'Mr. LePage',
  'him',
  'The governor',
  'he',
  'Mr. LePage',
  'his',
  'the governor',
  'his',
  'Mr. LePage’s',
  'him',
  'Mr. LePage’s',
  'the governor',
  'Mr. LePage',
  'Mr. LePage',
  'Mr. LePage’s',
  'the governor’s',
  'Mr. LePage’s',
  'he',
  'his',
  'his',
  'Mr. LePage',
  'I',
  'Mr. LePage',
  'his',
  'I',
  'Mr. LePage',
  'he',
  'Mr. LePage',
  'he',
  'he',
  'I',
  'Mr. LePage',
  'he'],
 ['a profane threat and generalizations about drugs and race that had prompted him to hint on Tuesday that he might abort his second term',
  'his threat',
  'his lates

In [ ]:
def create_one_list(nested_list)-> list:
    return [tup for sublist in nested_list for tup in sublist]

sorted_span = sorted(create_one_list(spans), key=lambda x: x[1], reverse=True)


for span in sorted_span:
    subject = [key for key, tuples in my_dict.items() if span in tuples][0]
    
    subject = ''
    for key, tuples in my_dict.items():
        if span in tuples:
            subject = key
            break
    
    start , end = span
    text = text[:start] + subject + text[end:]    
    
text

In [ ]:
span_index: list[str] = []
for index_list in spans:
    span_index = span_index + index_list
    
span_index.sort(reverse=True)

In [ ]:

if (1479, 1499) in span_index:
    print(True)

True
